# 📍 MEMBER 1 WORKSPACE: EDA & Candidate Feature Generation
**Owner: Member 1 (Data & Features)**

### Your Deliverables for the Team:
1. Preprocessing analysis and missing values.
2. Candidate and feature table: `[trip_id, obs_idx, arc_id, distance]`.
3. Candidate miss rate (percentage of observations where true road arc is NOT in candidates).
4. Trajectory plots & Initial Folium interactive map for **Report Section 2.2**.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium

print('✅ Libraries loaded successfully!')


### Step 1: Inspect Sample Trip Files


In [ ]:
DATA_DIR = '../data/student_test'
trip_id = '00000000'
trip_path = os.path.join(DATA_DIR, trip_id)

# 1. Load GPS observations (.track: lat, lon, time)
track_df = pd.read_csv(f'{trip_path}/{trip_id}.track', sep=r'\s+', header=None, names=['lat', 'lon', 'timestamp'])

# 2. Load road nodes (.nodes: lat, lon)
nodes_df = pd.read_csv(f'{trip_path}/{trip_id}.nodes', sep=r'\s+', header=None, names=['lat', 'lon'])

# 3. Load road arcs (.arcs: from_node, to_node)
arcs_df = pd.read_csv(f'{trip_path}/{trip_id}.arcs', sep=r'\s+', header=None, names=['from_node', 'to_node'])

# 4. Load ground truth route (.route: arc_id)
route_df = pd.read_csv(f'{trip_path}/{trip_id}.route', header=None, names=['arc_id'])

print(f'Trip {trip_id} Summary:')
print(f'  GPS Observations: {len(track_df)}')
print(f'  Road Nodes:       {len(nodes_df)}')
print(f'  Road Arcs:        {len(arcs_df)}')
print(f'  Ground Truth Arcs:{len(route_df)}')
track_df.head()


### Step 2: Missing Value & Sampling Interval Analysis (For Report 2.2)


In [ ]:
print('--- Missing Values Check ---')
print('Track missing:', track_df.isnull().sum().to_dict())
print('Nodes missing:', nodes_df.isnull().sum().to_dict())
print('Arcs missing: ', arcs_df.isnull().sum().to_dict())

time_diffs = track_df['timestamp'].diff().dropna()
print(f'Mean sampling interval: {time_diffs.mean():.2f}s (Min: {time_diffs.min()}s, Max: {time_diffs.max()}s)')


### Step 3: Haversine Distance Calculation


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0  # Earth radius in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2.0)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2.0)**2
    return 2.0 * R * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))

print('Haversine distance ready!')


### Step 4: Initial Folium Map (For Report Section 2.2)


In [ ]:
center_lat, center_lon = track_df['lat'].mean(), track_df['lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

for _, row in track_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color='blue',
        fill=True
    ).add_to(m)

os.makedirs('../maps', exist_ok=True)
m.save('../maps/01_eda_initial_map.html')
print('✅ Map saved to ../maps/01_eda_initial_map.html')
m
